# DINOv3 hierarchical classifier, variant A: linear head on frozen features

Two logits on one linear layer, both trained with BCE on cached DINOv3 features:
1. **Stage 1:** is this a problem? Obstacle and SurfaceProblem are problems; everything else is clean.
2. **Stage 2:** given a problem, is it an Obstacle? Trained and scored on true problem images only.

Reads a processed dataset built by `scripts/build_proc_sidewalk_data_A.py`, evaluates on `pitt-val`, and never loads
`pitt-test`. Needs a GPU runtime and an `HF_TOKEN` Colab secret. Results are on the Colab VM and vanish with the
runtime, so download `RUN_DIR` before releasing it.

In [1]:
!pip install -q -U transformers

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (switch runtime to GPU)")

torch 2.11.0+cu130 | transformers 5.22.1
GPU: NVIDIA A100-SXM4-40GB


## Config

`BASE` holds the defaults. `SWEEPS` is a list of dicts of ranges: each dict expands to all its combinations, and any
`BASE` key can be swept by adding it to a dict. `TRAIN_PLANS` lists `(split, steps)` phases, run in order.

In [39]:
from pathlib import Path

# Data
DATA_REPO = "gvadhul/sidewalk-proc-A-naive-merged-nulls"                      # private Hugging Face dataset repo holding DATA_TAR, e.g. "user/sidewalk-proc"
DATA_TAR = "proc-sidewalk-data-naive-merged-nulls.tar"
DATA_ROOT = Path("/content/proc-sidewalk-data-naive-merged-nulls")
CACHE_BATCH_SIZE = 256
RUN_DIR = Path("/content/dino_ft_A_runs")

# Labels: decided here, not in the processed dataset
PROBLEM_CLASSES = ["Obstacle", "SurfaceProblem"]
CLEAN_CLASSES = ["Crosswalk", "CurbRamp", "no_obstacles", "no_obstacles_manual_collection"]
EVAL_SPLIT = "pitt-val"
EVAL_DROP = []
K = 44                                # stage 1 Recall@K
THRESHOLD = 0.5

TRAIN_PLANS = {
    "pitt": [("pitt-train", 2000)],
    "other_then_pitt": [("other-cities-train", 2000), ("pitt-train", 2000)],
}

BASE = dict(
    base_model="facebook/dinov3-vitb16-pretrain-lvd1689m",
    pooling="cls",                    # "cls", "mean" (of patch tokens) or "cls+mean"
    train="pitt",                     # a TRAIN_PLANS key
    lr=1e-4,
    batch_size=64,
    eval_every=250,
    seed=0,
)

SWEEPS = [
    dict(train=["pitt", "other_then_pitt"], lr=[1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2], seed=[0, 1, 2], threshold=[0.3, 0.5, 0.7]),
]

In [40]:
from itertools import product

def expand(sweeps, base):
    return [{**base, **dict(zip(s, values))} for s in sweeps for values in product(*s.values())]

CONFIGS = expand(SWEEPS, BASE)
print(len(CONFIGS), "runs")

126 runs


## Data

In [ ]:
import tarfile, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from huggingface_hub import hf_hub_download
from PIL import Image
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except ImportError:
    HF_TOKEN = None

# drive.mount() kills the kernel under the Colab VS Code extension, so the dataset comes from Hugging Face.
if not DATA_ROOT.exists():
    t0 = time.perf_counter()
    tar = hf_hub_download(DATA_REPO, DATA_TAR, repo_type="dataset", token=HF_TOKEN)
    with tarfile.open(tar) as f:
        f.extractall(DATA_ROOT.parent, filter="data")
    print(f"downloaded and extracted in {time.perf_counter() - t0:.0f}s")

def load_split(split):
    df = pd.read_csv(DATA_ROOT / f"{split}.csv", low_memory=False)[["filename", "class", "image_path", "split", "is_frame", "source_class"]]
    if split == EVAL_SPLIT:
        df = df[~df["class"].isin(EVAL_DROP)].reset_index(drop=True)
    unknown = set(df["class"]) - set(PROBLEM_CLASSES) - set(CLEAN_CLASSES)
    assert not unknown, unknown
    df["is_problem"] = df["class"].isin(PROBLEM_CLASSES)
    df["is_obstacle"] = df["class"].eq("Obstacle")
    return df

def read_images(df):
    def read(path):
        with Image.open(DATA_ROOT / path) as im:
            return np.asarray(im)
    with ThreadPoolExecutor(16) as pool:
        return torch.from_numpy(np.stack(list(pool.map(read, df.image_path)))).permute(0, 3, 1, 2) # TODO: What does this do exactly? 

t0 = time.perf_counter()
SPLITS = sorted({split for plan in TRAIN_PLANS.values() for split, _ in plan} | {EVAL_SPLIT})
data = {s: load_split(s) for s in SPLITS}
images = {s: read_images(df) for s, df in data.items()}
print(f"read in {time.perf_counter() - t0:.0f}s")
for s, df in data.items():
    print(f"{s}: {len(df)} images, {(~df.is_problem).sum()} clean, {df.is_problem.sum()} problem, "
          f"{df.is_obstacle.sum()} Obstacle")

read in 18s
other-cities-train: 22912 images, 12015 clean, 10897 problem, 2376 Obstacle
pitt-train: 999 images, 571 clean, 428 problem, 39 Obstacle
pitt-val: 382 images, 244 clean, 138 problem, 14 Obstacle


## Frozen features

In [45]:
from torch import nn
from transformers import AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)  # ImageNet stats
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

@torch.inference_mode()
def embed(model, x):
    """CLS token and mean patch token per image, both cached so pooling can be swept without re-embedding."""
    cls, mean = [], []
    for i in range(0, len(x), CACHE_BATCH_SIZE):
        batch = (x[i:i + CACHE_BATCH_SIZE].to(device).float() / 255 - MEAN) / STD
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_amp):
            hs = model(pixel_values=batch).last_hidden_state
        cls.append(hs[:, 0].float())
        mean.append(hs[:, 1 + model.config.num_register_tokens:].mean(1).float())
    return dict(cls=torch.cat(cls), mean=torch.cat(mean))

features = {}
for name in {c["base_model"] for c in CONFIGS}:
    model = AutoModel.from_pretrained(name, token=HF_TOKEN).to(device).eval().requires_grad_(False)
    t0 = time.perf_counter()
    features[name] = {s: embed(model, x) for s, x in images.items()}
    print(f"{name}: {sum(map(len, images.values())) / (time.perf_counter() - t0):.0f} images/s")
    del model
del images
if device == "cuda":
    torch.cuda.empty_cache()

def pooled(cfg, split):
    f = features[cfg["base_model"]][split]
    return torch.cat([f[p] for p in cfg["pooling"].split("+")], 1)

def targets(split):
    df = data[split]
    return (torch.tensor(df.is_problem.values, device=device).float(),
            torch.tensor(df.is_obstacle.values, device=device).float())

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

facebook/dinov3-vitb16-pretrain-lvd1689m: 1937 images/s


## Metrics

Stage 1 ranks val images by P(problem). Recall@K can't exceed K / number of problems, so the cap is reported too.
Stage 2 uses true problem images only, so stage 1 errors don't affect it. In a single-label problem, a class's
accuracy is its recall, so per-class accuracy isn't listed separately. `pipeline_cm` runs both stages together.

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             precision_recall_fscore_support)

CLEAN, OBSTACLE, SURFACE = "clean", "Obstacle", "SurfaceProblem"

def evaluate(df, p_problem, p_obstacle, k=K, stage_1_threshold=THRESHOLD, stage_2_threshold = THRESHOLD):
    y = df.is_problem.values
    hits = y[np.argsort(-p_problem, kind="stable")[:k]].sum() # Number of problems detected in the top K most confident predictions
    pred_problem = p_problem >= stage_1_threshold
    m = {f"recall@{k}": hits / y.sum(), f"precision@{k}": hits / k, f"recall@{k}_cap": min(k, y.sum()) / y.sum(),
         "s1_accuracy": accuracy_score(y, pred_problem), "s1_balanced_accuracy": balanced_accuracy_score(y, pred_problem)}

    true_obs, pred_obs = df.is_obstacle.values[y], p_obstacle[y] >= stage_2_threshold
    p, r, _, _ = precision_recall_fscore_support(true_obs, pred_obs, labels=[True, False], zero_division=0)
    m |= {"s2_accuracy": accuracy_score(true_obs, pred_obs), "s2_balanced_accuracy": balanced_accuracy_score(true_obs, pred_obs),
          "s2_macro_precision": p.mean(), "s2_macro_recall": r.mean(), "s2_obstacle_precision": p[0],
          "s2_obstacle_recall": r[0], "s2_surface_precision": p[1], "s2_surface_recall": r[1]}

    names = [CLEAN, OBSTACLE, SURFACE]
    true3 = np.where(~y, CLEAN, np.where(df.is_obstacle.values, OBSTACLE, SURFACE))
    pred3 = np.where(~pred_problem, CLEAN, np.where(p_obstacle >= stage_2_threshold, OBSTACLE, SURFACE))
    confusion = dict(
        s1_cm=confusion_matrix(y, pred_problem, labels=[False, True]),
        s2_cm=confusion_matrix(true_obs, pred_obs, labels=[True, False]),
        pipeline_cm=confusion_matrix(true3, pred3, labels=names),
        s1_by_class=pd.crosstab(df["class"], pred_problem).reindex(columns=[False, True], fill_value=0).values,
        s1_by_class_names=sorted(df["class"].unique()))
    return m, confusion

## Training

In [17]:
def batches(n, steps, batch_size, generator):
    """Fixed number of batches, drawn without replacement and reshuffled once the split runs out."""
    perm, i = torch.randperm(n, generator=generator), 0
    for _ in range(steps):
        if i + batch_size > n:
            perm, i = torch.randperm(n, generator=generator), 0
        yield perm[i:i + batch_size].to(device)
        i += batch_size

def hierarchical_loss(logits, y_problem, y_obstacle):
    stage1 = nn.functional.binary_cross_entropy_with_logits(logits[:, 0], y_problem)
    mask = y_problem > 0
    if not mask.any():
        return stage1
    return stage1 + nn.functional.binary_cross_entropy_with_logits(logits[mask, 1], y_obstacle[mask])

@torch.inference_mode()
def predict(head, x):
    p = torch.sigmoid(head(x)).cpu().numpy()
    return p[:, 0], p[:, 1]

def run(cfg):
    torch.manual_seed(cfg["seed"])
    generator = torch.Generator().manual_seed(cfg["seed"])
    head = nn.Linear(pooled(cfg, EVAL_SPLIT).shape[1], 2).to(device)
    opt = torch.optim.Adam(head.parameters(), lr=cfg["lr"])
    val_x, val_df = pooled(cfg, EVAL_SPLIT), data[EVAL_SPLIT]
    plan = TRAIN_PLANS[cfg["train"]]
    total, step, rows = sum(n for _, n in plan), 0, []
    for split, steps in plan:
        x, (y_problem, y_obstacle) = pooled(cfg, split), targets(split)
        for idx in batches(len(x), steps, cfg["batch_size"], generator):
            loss = hierarchical_loss(head(x[idx]), y_problem[idx], y_obstacle[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            step += 1
            if step % cfg["eval_every"] == 0 or step == total:
                p_problem, p_obstacle = predict(head, val_x)
                m, confusion = evaluate(val_df, p_problem, p_obstacle)
                rows.append({**cfg, "step": step, "phase": split, "train_loss": loss.item(), **m})
    return rows, confusion, p_problem, p_obstacle

## Sweep

In [18]:
import json
from datetime import datetime, timezone

RUN_DIR = RUN_DIR / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR.mkdir(parents=True)
(RUN_DIR / "config.json").write_text(json.dumps(dict(base=BASE, sweeps=SWEEPS, train_plans=TRAIN_PLANS, data_tar=DATA_TAR,
                                                     k=K, threshold=THRESHOLD), indent=2))
history, confusions, predictions = [], {}, []
t0 = time.perf_counter()
for i, cfg in enumerate(CONFIGS):
    rows, confusion, p_problem, p_obstacle = run(cfg)
    history += rows
    key = (cfg["train"], cfg["lr"], cfg["seed"])
    confusions[key] = confusion
    predictions.append(data[EVAL_SPLIT][["filename", "class"]].assign(train=cfg["train"], lr=cfg["lr"], seed=cfg["seed"],
                                                                     p_problem=p_problem, p_obstacle=p_obstacle))
    print(f"[{i + 1}/{len(CONFIGS)}] {key}: precision@{K} {rows[-1][f'precision@{K}']:.3f}, "
          f"s2 balanced {rows[-1]['s2_balanced_accuracy']:.3f}, {time.perf_counter() - t0:.0f}s")
history = pd.DataFrame(history)
history.to_csv(RUN_DIR / "metrics.csv", index=False)
pd.concat(predictions).to_csv(RUN_DIR / "val_predictions.csv", index=False)
print("saved to", RUN_DIR)

[1/30] ('pitt', 0.0001, 0): precision@44 0.864, s2 balanced 0.532, 4s
[2/30] ('pitt', 0.0001, 1): precision@44 0.864, s2 balanced 0.563, 7s
[3/30] ('pitt', 0.0001, 2): precision@44 0.841, s2 balanced 0.496, 11s
[4/30] ('pitt', 0.0003, 0): precision@44 0.886, s2 balanced 0.567, 15s
[5/30] ('pitt', 0.0003, 1): precision@44 0.864, s2 balanced 0.563, 18s
[6/30] ('pitt', 0.0003, 2): precision@44 0.841, s2 balanced 0.567, 22s
[7/30] ('pitt', 0.001, 0): precision@44 0.795, s2 balanced 0.595, 25s
[8/30] ('pitt', 0.001, 1): precision@44 0.795, s2 balanced 0.591, 29s
[9/30] ('pitt', 0.001, 2): precision@44 0.795, s2 balanced 0.595, 32s
[10/30] ('pitt', 0.003, 0): precision@44 0.795, s2 balanced 0.591, 36s
[11/30] ('pitt', 0.003, 1): precision@44 0.795, s2 balanced 0.591, 39s
[12/30] ('pitt', 0.003, 2): precision@44 0.795, s2 balanced 0.591, 43s
[13/30] ('pitt', 0.01, 0): precision@44 0.795, s2 balanced 0.591, 46s
[14/30] ('pitt', 0.01, 1): precision@44 0.773, s2 balanced 0.591, 50s
[15/30] ('pit

## Results

Means over seeds at the final step, then how Recall@K moves during training. With 14 Obstacle images in val, stage 2
differences of a few points are noise, so compare the seed standard deviations before reading anything into them.

In [9]:
pd.set_option("display.width", 200, "display.max_columns", 40)
METRICS = [f"recall@{K}", f"precision@{K}", "s1_accuracy", "s1_balanced_accuracy", "s2_accuracy", "s2_balanced_accuracy",
           "s2_macro_precision", "s2_macro_recall", "s2_obstacle_precision", "s2_obstacle_recall",
           "s2_surface_precision", "s2_surface_recall"]
final = history[history.groupby(["train", "lr", "seed"]).step.transform("max") == history.step]
print(f"recall@{K} cap: {final[f'recall@{K}_cap'].iloc[0]:.3f}")
summary = final.groupby(["train", "lr"])[METRICS].agg(["mean", "std"]).round(3)
summary.to_csv(RUN_DIR / "summary.csv")
for m in METRICS:
    print(f"\n{m}, mean (std) over seeds")
    t = summary[m]
    print((t["mean"].map("{:.3f}".format) + " (" + t["std"].map("{:.3f}".format) + ")").unstack("lr").to_string())

recall@40 cap: 0.290

recall@40, mean (std) over seeds
lr                      0.0001         0.0003         0.0010         0.0030         0.0100
train                                                                                     
other_then_pitt  0.251 (0.004)  0.254 (0.000)  0.239 (0.000)  0.234 (0.004)  0.225 (0.000)
pitt             0.256 (0.004)  0.263 (0.004)  0.232 (0.000)  0.232 (0.007)  0.229 (0.004)

precision@40, mean (std) over seeds
lr                      0.0001         0.0003         0.0010         0.0030         0.0100
train                                                                                     
other_then_pitt  0.867 (0.014)  0.875 (0.000)  0.825 (0.000)  0.808 (0.014)  0.775 (0.000)
pitt             0.883 (0.014)  0.908 (0.014)  0.800 (0.000)  0.800 (0.025)  0.792 (0.014)

s1_accuracy, mean (std) over seeds
lr                      0.0001         0.0003         0.0010         0.0030         0.0100
train                                                

In [10]:
print(f"Mean precision@{K} over seeds, by step")
print(history.groupby(["train", "lr", "step"])[f"precision@{K}"].mean().unstack("step").round(3).to_string())
print("\nMean stage 2 balanced accuracy over seeds, by step")
print(history.groupby(["train", "lr", "step"])["s2_balanced_accuracy"].mean().unstack("step").round(3).to_string())

Mean precision@40 over seeds, by step
step                     250    500    750    1000   1250   1500   1750   2000   2250   2500   2750   3000   3250   3500   3750   4000
train           lr                                                                                                                    
other_then_pitt 0.0001  0.458  0.500  0.517  0.542  0.550  0.550  0.558  0.575  0.717  0.833  0.875  0.875  0.875  0.875  0.867  0.867
                0.0003  0.508  0.558  0.583  0.575  0.575  0.592  0.575  0.608  0.875  0.875  0.867  0.858  0.858  0.867  0.875  0.875
                0.0010  0.567  0.583  0.625  0.617  0.633  0.650  0.625  0.667  0.875  0.875  0.850  0.850  0.825  0.825  0.825  0.825
                0.0030  0.625  0.675  0.667  0.633  0.667  0.675  0.625  0.667  0.850  0.817  0.800  0.800  0.800  0.800  0.808  0.808
                0.0100  0.633  0.675  0.700  0.617  0.692  0.667  0.617  0.633  0.850  0.817  0.800  0.800  0.783  0.775  0.783  0.775
pitt            0

Confusion matrices, summed over seeds at the final step. Rows are truth. Set `SHOW` to any `(train, lr)`.

In [11]:
def show_confusion(train, lr):
    keys = [k for k in confusions if k[:2] == (train, lr)]
    total = lambda name: sum(confusions[k][name] for k in keys)
    print(f"\n=== {train}, lr={lr}, {len(keys)} seeds ===")
    print(pd.DataFrame(total("s1_cm"), ["clean", "problem"], ["pred clean", "pred problem"]).to_string())
    print(pd.DataFrame(total("s1_by_class"), confusions[keys[0]]["s1_by_class_names"], ["pred clean", "pred problem"]).to_string())
    print(pd.DataFrame(total("s2_cm"), ["Obstacle", "SurfaceProblem"], ["pred Obstacle", "pred SurfaceProblem"]).to_string())
    print(pd.DataFrame(total("pipeline_cm"), [CLEAN, OBSTACLE, SURFACE], ["pred " + n for n in (CLEAN, OBSTACLE, SURFACE)]).to_string())

SHOW = [(train, final[final.train == train].groupby("lr")[f"precision@{K}"].mean().idxmax()) for train in final.train.unique()]
for train, lr in SHOW:
    show_confusion(train, lr)


=== pitt, lr=0.0003, 3 seeds ===
         pred clean  pred problem
clean           569           163
problem         117           297
                pred clean  pred problem
Crosswalk               28             5
CurbRamp               391            65
Obstacle                18            24
SurfaceProblem          99           273
no_obstacles           150            93
                pred Obstacle  pred SurfaceProblem
Obstacle                    6                   36
SurfaceProblem              4                  368
                pred clean  pred Obstacle  pred SurfaceProblem
clean                  569              4                  159
Obstacle                18              3                   21
SurfaceProblem          99              3                  270

=== other_then_pitt, lr=0.0003, 3 seeds ===
         pred clean  pred problem
clean           567           165
problem         108           306
                pred clean  pred problem
Crosswalk               3